## Proper Preprocessing

In [1]:
import os
import cv2
import dlib
import pickle
import numpy as np
from tqdm import tqdm
import gc

# --- CONFIGURATION ---
DATASET_PATH = "facesm5"
OUTPUT_PICKLE = "encodings.pickle"
CNN_FACE_DETECTOR = "mmod_human_face_detector.dat"
SHAPE_PREDICTOR = "shape_predictor_68_face_landmarks.dat"
FACE_RECOGNITION_MODEL = "dlib_face_recognition_resnet_model_v1.dat"

# Save to disk every N images to free memory
BATCH_SAVE_SIZE = 10  # Reduced for processing full-size images

def load_models():
    print("[INFO] Loading dlib models...")
    if dlib.DLIB_USE_CUDA:
        print("[INFO] CUDA enabled. Using GPU-accelerated CNN detector.")
        detector = dlib.cnn_face_detection_model_v1(CNN_FACE_DETECTOR)
    else:
        print("[WARNING] CUDA not found. Falling back to HOG.")
        detector = dlib.get_frontal_face_detector()

    predictor = dlib.shape_predictor(SHAPE_PREDICTOR)
    facerec = dlib.face_recognition_model_v1(FACE_RECOGNITION_MODEL)
    return detector, predictor, facerec

def save_batch(encodings, names, batch_num):
    """Save a batch to temporary file"""
    temp_file = f"encodings_batch_{batch_num}.tmp"
    data = {"encodings": encodings, "names": names}
    with open(temp_file, "wb") as f:
        f.write(pickle.dumps(data))
    return temp_file

def merge_batches(temp_files):
    """Merge all temporary batch files into final pickle"""
    print("[INFO] Merging batches...")
    all_encodings = []
    all_names = []

    for temp_file in temp_files:
        with open(temp_file, "rb") as f:
            data = pickle.loads(f.read())
            all_encodings.extend(data["encodings"])
            all_names.extend(data["names"])
        os.remove(temp_file)  # Clean up temp file

    return all_encodings, all_names

def process_dataset():
    detector, predictor, facerec = load_models()
    known_encodings = []
    known_names = []
    temp_files = []
    batch_num = 0

    image_paths = []
    for root, dirs, files in os.walk(DATASET_PATH):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_paths.append(os.path.join(root, file))

    print(f"[INFO] Processing {len(image_paths)} images at original resolution...")

    for idx, img_path in enumerate(tqdm(image_paths)):
        name = img_path.split(os.path.sep)[-2]

        image = cv2.imread(img_path)
        if image is None:
            continue

        # Process at original size - NO RESIZING
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        try:
            # CRITICAL: upsample=0 for GPU memory
            results = detector(rgb, 0)

            if len(results) == 0:
                # Explicitly delete to free memory immediately
                del rgb, image
                gc.collect()
                continue

            # Process only the FIRST face to save GPU memory
            # If you need all faces, change to: for result in results:
            result = results[0]  # Take only first face

            if isinstance(detector, dlib.cnn_face_detection_model_v1):
                rect = result.rect
            else:
                rect = result

            shape = predictor(rgb, rect)
            face_encoding = np.array(facerec.compute_face_descriptor(rgb, shape))

            known_encodings.append(face_encoding)
            known_names.append(name)

            # Force memory cleanup after each image
            del rgb, image, results, shape
            gc.collect()

            # Save batch and clear memory every BATCH_SAVE_SIZE images
            if (idx + 1) % BATCH_SAVE_SIZE == 0:
                temp_file = save_batch(known_encodings, known_names, batch_num)
                temp_files.append(temp_file)
                batch_num += 1

                # Clear the lists to free memory
                known_encodings = []
                known_names = []

                # Force garbage collection
                gc.collect()
                print(f"\n[INFO] Batch {batch_num} saved, memory cleared")

        except RuntimeError as e:
            print(f"\n[ERROR] GPU Out of Memory on {img_path}. Skipping.")
            # Aggressive cleanup on error
            try:
                del rgb, image
            except:
                pass
            gc.collect()
            continue

    # Save any remaining encodings
    if known_encodings:
        temp_file = save_batch(known_encodings, known_names, batch_num)
        temp_files.append(temp_file)

    # Merge all batches into final file
    if temp_files:
        all_encodings, all_names = merge_batches(temp_files)
        print(f"[INFO] Serializing {len(all_encodings)} encodings to {OUTPUT_PICKLE}...")
        data = {"encodings": all_encodings, "names": all_names}
        with open(OUTPUT_PICKLE, "wb") as f:
            f.write(pickle.dumps(data))
        print("[INFO] Done.")
    else:
        print("[WARNING] No encodings generated.")

if __name__ == "__main__":
    if os.path.exists(DATASET_PATH):
        process_dataset()
    else:
        print("Dataset not found.")

[INFO] Loading dlib models...
[INFO] CUDA enabled. Using GPU-accelerated CNN detector.
[INFO] Processing 43 images at original resolution...


 26%|██▌       | 11/43 [00:04<00:04,  7.22it/s]


[INFO] Batch 1 saved, memory cleared


 47%|████▋     | 20/43 [00:05<00:03,  7.23it/s]


[INFO] Batch 2 saved, memory cleared


...


[INFO] Batch 3 saved, memory cleared


 93%|█████████▎| 40/43 [00:17<00:01,  2.14it/s]


[INFO] Batch 4 saved, memory cleared


100%|██████████| 43/43 [00:17<00:00,  2.40it/s]


[INFO] Merging batches...
[INFO] Serializing 43 encodings to encodings.pickle...
[INFO] Done.


## Webcam Face Recognition

In [3]:
import cv2
import dlib
import pickle
import numpy as np
import time
import csv
import os
from datetime import datetime

# --- CONFIGURATION ---
ENCODINGS_FILE = "encodings.pickle"
RECOGNITION_THRESHOLD = 0.5
PROCESS_EVERY_N_FRAMES = 2
FRAME_WIDTH = 640

# Model Paths
CNN_FACE_DETECTOR = "mmod_human_face_detector.dat"
SHAPE_PREDICTOR = "shape_predictor_68_face_landmarks.dat"
FACE_RECOGNITION_MODEL = "dlib_face_recognition_resnet_model_v1.dat"

# Global set to keep track of who has already been marked present in this session
present_students = set()

def mark_attendance(name):
    """Logs the name and current time to a CSV file."""
    # If student is already marked present in this session, skip
    if name in present_students:
        return

    # Create filename based on today's date (e.g., attendance_2025-12-30.csv)
    now = datetime.now()
    date_str = now.strftime("%Y-%m-%d")
    filename = f"attendance_{date_str}.csv"
    time_str = now.strftime("%H:%M:%S")

    # Check if file exists to determine if we need to write headers
    file_exists = os.path.isfile(filename)

    with open(filename, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["Name", "Time", "Date"]) # Header

        writer.writerow([name, time_str, date_str])

    # Add to set so we don't log them again instantly
    present_students.add(name)
    print(f"[ATTENDANCE] Marked {name} at {time_str}")

def load_resources():
    print("[INFO] Loading encodings and models...")
    with open(ENCODINGS_FILE, "rb") as f:
        data = pickle.load(f)

    # Check if CUDA is active
    if dlib.DLIB_USE_CUDA:
        print("[INFO] CUDA is active. Using GPU.")
    else:
        print("[WARNING] CUDA is NOT active. Running on CPU.")

    detector = dlib.cnn_face_detection_model_v1(CNN_FACE_DETECTOR)
    predictor = dlib.shape_predictor(SHAPE_PREDICTOR)
    facerec = dlib.face_recognition_model_v1(FACE_RECOGNITION_MODEL)

    return data, detector, predictor, facerec

def run_inference():
    data, detector, predictor, facerec = load_resources()
    known_encodings = np.array(data["encodings"])
    known_names = data["names"]

    print("[INFO] Starting video stream...")
    video_capture = cv2.VideoCapture(0)

    if not video_capture.isOpened():
        print("[ERROR] Could not open video stream.")
        return

    frame_count = 0
    face_locations = []
    face_names = []

    # Variables for FPS calculation
    fps_start_time = time.time()
    fps_frame_counter = 0
    fps_display = 0

    while True:
        ret, frame = video_capture.read()
        if not ret:
            break

        # Resize
        h, w = frame.shape[:2]
        ratio = FRAME_WIDTH / float(w)
        new_h = int(h * ratio)
        frame_resized = cv2.resize(frame, (FRAME_WIDTH, new_h))
        rgb_frame = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)

        # Skip frames logic
        if frame_count % PROCESS_EVERY_N_FRAMES == 0:
            face_locations = []
            face_names = []

            # 1. Detect (GPU)
            results = detector(rgb_frame, 0)

            for result in results:
                rect = result.rect
                left, top, right, bottom = rect.left(), rect.top(), rect.right(), rect.bottom()
                face_locations.append((left, top, right, bottom))

                # 2. Encode
                shape = predictor(rgb_frame, rect)
                face_encoding = np.array(facerec.compute_face_descriptor(rgb_frame, shape))

                # 3. Match
                distances = np.linalg.norm(known_encodings - face_encoding, axis=1)
                min_distance_idx = np.argmin(distances)
                min_distance = distances[min_distance_idx]

                if min_distance < RECOGNITION_THRESHOLD:
                    name = known_names[min_distance_idx]
                    mark_attendance(name) # <--- Log to CSV here
                else:
                    name = "Unknown"

                face_names.append(name)

        frame_count += 1

        # --- FPS Calculation ---
        fps_frame_counter += 1
        if (time.time() - fps_start_time) > 1: # Update FPS every 1 second
            fps_display = fps_frame_counter / (time.time() - fps_start_time)
            fps_frame_counter = 0
            fps_start_time = time.time()

        # --- DRAWING ---
        for (left, top, right, bottom), name in zip(face_locations, face_names):
            scale = 1 / ratio
            left, top, right, bottom = int(left * scale), int(top * scale), int(right * scale), int(bottom * scale)

            # Color: Green for recognized, Red for Unknown
            color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)

            cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
            cv2.rectangle(frame, (left, bottom - 35), (right, bottom), color, cv2.FILLED)
            cv2.putText(frame, name, (left + 6, bottom - 6), cv2.FONT_HERSHEY_DUPLEX, 0.8, (255, 255, 255), 1)

        # Draw FPS counter
        cv2.putText(frame, f"FPS: {fps_display:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        cv2.imshow('Face Recognition (GPU)', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    video_capture.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    if os.path.exists(ENCODINGS_FILE):
        run_inference()
    else:
        print(f"[ERROR] {ENCODINGS_FILE} not found.")

[INFO] Loading encodings and models...
[INFO] CUDA is active. Using GPU.
[INFO] Starting video stream...
[ATTENDANCE] Marked Daulet at 23:06:42


In [5]:
import cv2
import dlib
import pickle
import numpy as np
import time
import csv
import os
from datetime import datetime

# --- CONFIGURATION ---
ENCODINGS_FILE = "encodings.pickle"
RECOGNITION_THRESHOLD = 0.5
PROCESS_EVERY_N_FRAMES = 2
FRAME_WIDTH = 640
SCREENSHOT_DIR = "attendance_screenshots"  # Folder to save images

# Create screenshot directory if it doesn't exist
if not os.path.exists(SCREENSHOT_DIR):
    os.makedirs(SCREENSHOT_DIR)

# Model Paths
CNN_FACE_DETECTOR = "mmod_human_face_detector.dat"
SHAPE_PREDICTOR = "shape_predictor_68_face_landmarks.dat"
FACE_RECOGNITION_MODEL = "dlib_face_recognition_resnet_model_v1.dat"

# Global set to keep track of who has already been marked present in this session
present_students = set()

def mark_attendance(name):
    """
    Logs the name to CSV.
    Returns True if this is a NEW marking (needs screenshot).
    Returns False if already marked.
    """
    if name in present_students:
        return False

    now = datetime.now()
    date_str = now.strftime("%Y-%m-%d")
    filename = f"attendance_{date_str}.csv"
    time_str = now.strftime("%H:%M:%S")

    file_exists = os.path.isfile(filename)

    with open(filename, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["Name", "Time", "Date"])

        writer.writerow([name, time_str, date_str])

    present_students.add(name)
    print(f"[ATTENDANCE] Marked {name} at {time_str}")
    return True

def load_resources():
    print("[INFO] Loading encodings and models...")
    with open(ENCODINGS_FILE, "rb") as f:
        data = pickle.load(f)

    if dlib.DLIB_USE_CUDA:
        print("[INFO] CUDA is active. Using GPU.")
    else:
        print("[WARNING] CUDA is NOT active. Running on CPU.")

    detector = dlib.cnn_face_detection_model_v1(CNN_FACE_DETECTOR)
    predictor = dlib.shape_predictor(SHAPE_PREDICTOR)
    facerec = dlib.face_recognition_model_v1(FACE_RECOGNITION_MODEL)

    return data, detector, predictor, facerec

def align_face(rgb_frame, rect, predictor):
    shape = predictor(rgb_frame, rect)
    # Use the landmarks to align face
    # Example of getting specific landmarks (for eyes, nose, etc.)
    landmarks = np.array([(p.x, p.y) for p in shape.parts()])

    # Here, you can implement your alignment logic using the eye positions, etc.
    # Align face to a frontal pose based on landmarks
    # Use dlib or OpenCV for affine transformations or similarity transformations
    return landmarks

def run_inference():
    data, detector, predictor, facerec = load_resources()
    known_encodings = np.array(data["encodings"])
    known_names = data["names"]

    print("[INFO] Starting video stream...")
    video_capture = cv2.VideoCapture(0)

    if not video_capture.isOpened():
        print("[ERROR] Could not open video stream.")
        return

    frame_count = 0
    face_locations = []
    face_names = []

    # List to track who was marked in the specific current frame
    newly_marked_in_this_frame = []

    fps_start_time = time.time()
    fps_frame_counter = 0
    fps_display = 0

    while True:
        ret, frame = video_capture.read()
        if not ret:
            break

        # Resize
        h, w = frame.shape[:2]
        ratio = FRAME_WIDTH / float(w)
        new_h = int(h * ratio)
        frame_resized = cv2.resize(frame, (FRAME_WIDTH, new_h))
        rgb_frame = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)

        # Reset the "newly marked" list for this frame
        newly_marked_in_this_frame = []

        # Skip frames logic
        if frame_count % PROCESS_EVERY_N_FRAMES == 0:
            face_locations = []
            face_names = []

            # 1. Detect
            results = detector(rgb_frame, 0)

            for result in results:
                rect = result.rect
                left, top, right, bottom = rect.left(), rect.top(), rect.right(), rect.bottom()
                face_locations.append((left, top, right, bottom))

                # 2. Encode
                shape = predictor(rgb_frame, rect)
                face_encoding = np.array(facerec.compute_face_descriptor(rgb_frame, shape))

                # 3. Match
                distances = np.linalg.norm(known_encodings - face_encoding, axis=1)
                min_distance_idx = np.argmin(distances)
                min_distance = distances[min_distance_idx]

                if min_distance < RECOGNITION_THRESHOLD:
                    name = known_names[min_distance_idx]

                    # LOGIC CHANGE: Check if this returns True
                    if mark_attendance(name):
                        newly_marked_in_this_frame.append(name)
                else:
                    name = "Unknown"

                face_names.append(name)

        frame_count += 1

        # --- FPS Calculation ---
        fps_frame_counter += 1
        if (time.time() - fps_start_time) > 1:
            fps_display = fps_frame_counter / (time.time() - fps_start_time)
            fps_frame_counter = 0
            fps_start_time = time.time()

        # --- DRAWING (Boxes are added to 'frame' here) ---
        for (left, top, right, bottom), name in zip(face_locations, face_names):
            scale = 1 / ratio
            left, top, right, bottom = int(left * scale), int(top * scale), int(right * scale), int(bottom * scale)

            color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)

            cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
            cv2.rectangle(frame, (left, bottom - 35), (right, bottom), color, cv2.FILLED)
            cv2.putText(frame, name, (left + 6, bottom - 6), cv2.FONT_HERSHEY_DUPLEX, 0.8, (255, 255, 255), 1)

        # --- SNAPSHOT SAVING LOGIC ---
        # We do this AFTER drawing, so the saved image includes the boxes
        if newly_marked_in_this_frame:
            for name in newly_marked_in_this_frame:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                filename = f"{name}_{timestamp}.jpg"
                filepath = os.path.join(SCREENSHOT_DIR, filename)

                # Save the frame that now has boxes drawn on it
                cv2.imwrite(filepath, frame)
                print(f"[SNAPSHOT] Saved evidence: {filepath}")

        # Draw FPS counter
        cv2.putText(frame, f"FPS: {fps_display:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        cv2.imshow('Face Recognition (GPU)', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    video_capture.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    if os.path.exists(ENCODINGS_FILE):
        run_inference()
    else:
        print(f"[ERROR] {ENCODINGS_FILE} not found.")

[INFO] Loading encodings and models...
[INFO] CUDA is active. Using GPU.
[INFO] Starting video stream...
[ATTENDANCE] Marked Daulet at 23:07:02
[SNAPSHOT] Saved evidence: attendance_screenshots\Daulet_20260121_230702.jpg
[ATTENDANCE] Marked Adyl at 23:07:14
[SNAPSHOT] Saved evidence: attendance_screenshots\Adyl_20260121_230714.jpg


In [3]:
import cv2
import dlib
import pickle
import numpy as np
import time
import csv
import os
from datetime import datetime
from collections import deque  # New import for the sliding window

# --- CONFIGURATION ---
ENCODINGS_FILE = "encodings.pickle"
RECOGNITION_THRESHOLD = 0.5
PROCESS_EVERY_N_FRAMES = 2
FRAME_WIDTH = 640
SCREENSHOT_DIR = "attendance_screenshots"

# --- CONSISTENCY CHECKS ---
HISTORY_SIZE = 30       # Total frames to look back
REQUIRED_HITS = 20      # How many times face must be seen in history to mark

if not os.path.exists(SCREENSHOT_DIR):
    os.makedirs(SCREENSHOT_DIR)

# Model Paths
CNN_FACE_DETECTOR = "mmod_human_face_detector.dat"
SHAPE_PREDICTOR = "shape_predictor_68_face_landmarks.dat"
FACE_RECOGNITION_MODEL = "dlib_face_recognition_resnet_model_v1.dat"

# Global set to keep track of who has already been marked present
present_students = set()

def mark_attendance(name):
    """
    Logs the name to CSV.
    Returns True if this is a NEW marking (needs screenshot).
    Returns False if already marked.
    """
    if name in present_students:
        return False

    now = datetime.now()
    date_str = now.strftime("%Y-%m-%d")
    filename = f"attendance_{date_str}.csv"
    time_str = now.strftime("%H:%M:%S")

    file_exists = os.path.isfile(filename)

    with open(filename, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["Name", "Time", "Date"])

        writer.writerow([name, time_str, date_str])

    present_students.add(name)
    print(f"[ATTENDANCE] Verified and Marked {name} at {time_str}")
    return True

def load_resources():
    print("[INFO] Loading encodings and models...")
    with open(ENCODINGS_FILE, "rb") as f:
        data = pickle.load(f)

    if dlib.DLIB_USE_CUDA:
        print("[INFO] CUDA is active. Using GPU.")
    else:
        print("[WARNING] CUDA is NOT active. Running on CPU.")

    detector = dlib.cnn_face_detection_model_v1(CNN_FACE_DETECTOR)
    predictor = dlib.shape_predictor(SHAPE_PREDICTOR)
    facerec = dlib.face_recognition_model_v1(FACE_RECOGNITION_MODEL)

    return data, detector, predictor, facerec

def run_inference():
    data, detector, predictor, facerec = load_resources()
    known_encodings = np.array(data["encodings"])
    known_names = data["names"]

    print("[INFO] Starting video stream...")
    video_capture = cv2.VideoCapture(0)

    if not video_capture.isOpened():
        print("[ERROR] Could not open video stream.")
        return

    frame_count = 0
    face_locations = []
    face_names = []

    # Initialize the sliding window
    # This stores sets of names found in the last N processed frames
    recognition_history = deque(maxlen=HISTORY_SIZE)

    newly_marked_in_this_frame = []

    # For UI display only (to show progress like "15/20")
    current_verification_stats = {}

    fps_start_time = time.time()
    fps_frame_counter = 0
    fps_display = 0

    while True:
        ret, frame = video_capture.read()
        if not ret:
            break

        # Resize
        h, w = frame.shape[:2]
        ratio = FRAME_WIDTH / float(w)
        new_h = int(h * ratio)
        frame_resized = cv2.resize(frame, (FRAME_WIDTH, new_h))
        rgb_frame = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)

        newly_marked_in_this_frame = []

        # Skip frames logic
        if frame_count % PROCESS_EVERY_N_FRAMES == 0:
            face_locations = []
            face_names = []

            # Names found in *this* specific frame
            names_in_current_frame = set()

            # 1. Detect
            results = detector(rgb_frame, 0)

            for result in results:
                rect = result.rect
                left, top, right, bottom = rect.left(), rect.top(), rect.right(), rect.bottom()
                face_locations.append((left, top, right, bottom))

                # 2. Encode
                shape = predictor(rgb_frame, rect)
                face_encoding = np.array(facerec.compute_face_descriptor(rgb_frame, shape))

                # 3. Match
                distances = np.linalg.norm(known_encodings - face_encoding, axis=1)
                min_distance_idx = np.argmin(distances)
                min_distance = distances[min_distance_idx]

                if min_distance < RECOGNITION_THRESHOLD:
                    name = known_names[min_distance_idx]
                    names_in_current_frame.add(name)
                else:
                    name = "Unknown"

                face_names.append(name)

            # --- HISTORY LOGIC ---
            # Append the set of names found in this frame to history
            recognition_history.append(names_in_current_frame)

            # Check consistency for every name found in this frame
            current_verification_stats = {} # Reset stats for UI

            for name in names_in_current_frame:
                if name == "Unknown":
                    continue

                # Count how many times this name appears in the detection history
                # history is a deque of sets: [{'Bob'}, {'Bob', 'Alice'}, ...]
                occurrence_count = sum(1 for frame_set in recognition_history if name in frame_set)

                # Update UI stats
                current_verification_stats[name] = occurrence_count

                # If threshold met AND not already marked
                if occurrence_count >= REQUIRED_HITS:
                    if mark_attendance(name):
                        newly_marked_in_this_frame.append(name)

        frame_count += 1

        # --- FPS Calculation ---
        fps_frame_counter += 1
        if (time.time() - fps_start_time) > 1:
            fps_display = fps_frame_counter / (time.time() - fps_start_time)
            fps_frame_counter = 0
            fps_start_time = time.time()

        # --- DRAWING ---
        for (left, top, right, bottom), name in zip(face_locations, face_names):
            scale = 1 / ratio
            left, top, right, bottom = int(left * scale), int(top * scale), int(right * scale), int(bottom * scale)

            # Determine color and label based on state
            if name == "Unknown":
                color = (0, 0, 255) # Red
                label = name
            elif name in present_students:
                color = (0, 255, 0) # Green (Already Marked)
                label = f"{name} (Present)"
            else:
                color = (0, 255, 255) # Yellow (Verifying...)
                # Get stats from our logic above
                hits = current_verification_stats.get(name, 0)
                label = f"{name} {hits}/{REQUIRED_HITS}"

            cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
            cv2.rectangle(frame, (left, bottom - 35), (right, bottom), color, cv2.FILLED)
            cv2.putText(frame, label, (left + 6, bottom - 6), cv2.FONT_HERSHEY_DUPLEX, 0.6, (0, 0, 0), 1)

        # --- SNAPSHOT SAVING LOGIC ---
        if newly_marked_in_this_frame:
            for name in newly_marked_in_this_frame:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                filename = f"{name}_{timestamp}.jpg"
                filepath = os.path.join(SCREENSHOT_DIR, filename)
                cv2.imwrite(filepath, frame)
                print(f"[SNAPSHOT] Saved evidence: {filepath}")

        cv2.putText(frame, f"FPS: {fps_display:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        cv2.imshow('Face Recognition (GPU)', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    video_capture.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    if os.path.exists(ENCODINGS_FILE):
        run_inference()
    else:
        print(f"[ERROR] {ENCODINGS_FILE} not found.")

[INFO] Loading encodings and models...
[INFO] CUDA is active. Using GPU.
[INFO] Starting video stream...
[ATTENDANCE] Verified and Marked Daulet at 23:16:15
[SNAPSHOT] Saved evidence: attendance_screenshots\Daulet_20260127_231615.jpg
[ATTENDANCE] Verified and Marked Adyl at 23:16:21
[SNAPSHOT] Saved evidence: attendance_screenshots\Adyl_20260127_231621.jpg


# GPT version

In [1]:
import cv2
import dlib
import pickle
import numpy as np
import time
import csv
import os
from datetime import datetime
from collections import deque

# --- CONFIGURATION ---
ENCODINGS_FILE = "encodings.pickle"
RECOGNITION_THRESHOLD = 0.5
PROCESS_EVERY_N_FRAMES = 2
FRAME_WIDTH = 640
SCREENSHOT_DIR = "attendance_screenshots"

# --- CONSISTENCY CHECKS ---
HISTORY_SIZE = 30       # Total frames to look back
REQUIRED_HITS = 20      # How many times face must be seen in history to mark

if not os.path.exists(SCREENSHOT_DIR):
    os.makedirs(SCREENSHOT_DIR)

# Model Paths
CNN_FACE_DETECTOR = "mmod_human_face_detector.dat"
SHAPE_PREDICTOR = "shape_predictor_68_face_landmarks.dat"
FACE_RECOGNITION_MODEL = "dlib_face_recognition_resnet_model_v1.dat"

# Global set to keep track of who has already been marked present
present_students = set()

# --- HELPER FUNCTION: AFFINE ALIGNMENT ---
def align_face(image, shape, desired_face_width=150, desired_left_eye=(0.35, 0.35)):
    """
     aligns the face based on eye coordinates to handle head tilt/rotation.
     desired_face_width=150 is dlib's preferred input size.
    """
    # 1. Extract eye coordinates based on 68-point or 5-point model
    if shape.num_parts == 68:
        l_eye_pts = [(shape.part(i).x, shape.part(i).y) for i in range(36, 42)]
        r_eye_pts = [(shape.part(i).x, shape.part(i).y) for i in range(42, 48)]
    else:
        # Fallback for 5-point model
        l_eye_pts = [(shape.part(i).x, shape.part(i).y) for i in range(2, 4)]
        r_eye_pts = [(shape.part(i).x, shape.part(i).y) for i in range(0, 2)]

    # 2. Compute center of mass for each eye
    left_eye_center = np.mean(l_eye_pts, axis=0).astype("int")
    right_eye_center = np.mean(r_eye_pts, axis=0).astype("int")

    # 3. Compute angle between eyes
    dY = right_eye_center[1] - left_eye_center[1]
    dX = right_eye_center[0] - left_eye_center[0]
    angle = np.degrees(np.arctan2(dY, dX)) - 180

    # 4. Calculate scale
    desired_right_eye_x = 1.0 - desired_left_eye[0]
    dist = np.sqrt((dX ** 2) + (dY ** 2))
    desired_dist = (desired_right_eye_x - desired_left_eye[0]) * desired_face_width
    scale = desired_dist / dist

    # 5. Compute center (between eyes)
    center_x = int((left_eye_center[0] + right_eye_center[0]) // 2)
    center_y = int((left_eye_center[1] + right_eye_center[1]) // 2)
    eyes_center = (center_x, center_y)
    # 6. Get Rotation Matrix
    M = cv2.getRotationMatrix2D(eyes_center, angle, scale)

    # 7. Update translation to center eyes in the new image
    tX = desired_face_width * 0.5
    tY = desired_face_width * desired_left_eye[1]
    M[0, 2] += (tX - eyes_center[0])
    M[1, 2] += (tY - eyes_center[1])

    # 8. Warp (Align)
    output = cv2.warpAffine(image, M, (desired_face_width, desired_face_width),
                            flags=cv2.INTER_CUBIC)

    return output

def mark_attendance(name):
    if name in present_students:
        return False

    now = datetime.now()
    date_str = now.strftime("%Y-%m-%d")
    filename = f"attendance_{date_str}.csv"
    time_str = now.strftime("%H:%M:%S")

    file_exists = os.path.isfile(filename)

    with open(filename, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["Name", "Time", "Date"])
        writer.writerow([name, time_str, date_str])

    present_students.add(name)
    print(f"[ATTENDANCE] Verified and Marked {name} at {time_str}")
    return True

def load_resources():
    print("[INFO] Loading encodings and models...")
    with open(ENCODINGS_FILE, "rb") as f:
        data = pickle.load(f)

    if dlib.DLIB_USE_CUDA:
        print("[INFO] CUDA is active. Using GPU.")
    else:
        print("[WARNING] CUDA is NOT active. Running on CPU.")

    detector = dlib.cnn_face_detection_model_v1(CNN_FACE_DETECTOR)
    predictor = dlib.shape_predictor(SHAPE_PREDICTOR)
    facerec = dlib.face_recognition_model_v1(FACE_RECOGNITION_MODEL)

    return data, detector, predictor, facerec

def run_inference():
    data, detector, predictor, facerec = load_resources()
    known_encodings = np.array(data["encodings"])
    known_names = data["names"]

    print("[INFO] Starting video stream...")
    video_capture = cv2.VideoCapture(0)

    if not video_capture.isOpened():
        print("[ERROR] Could not open video stream.")
        return

    frame_count = 0
    face_locations = []
    face_names = []
    recognition_history = deque(maxlen=HISTORY_SIZE)
    newly_marked_in_this_frame = []
    current_verification_stats = {}

    fps_start_time = time.time()
    fps_frame_counter = 0
    fps_display = 0

    while True:
        ret, frame = video_capture.read()
        if not ret:
            break

        # Resize
        h, w = frame.shape[:2]
        ratio = FRAME_WIDTH / float(w)
        new_h = int(h * ratio)
        frame_resized = cv2.resize(frame, (FRAME_WIDTH, new_h))
        rgb_frame = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)

        newly_marked_in_this_frame = []

        if frame_count % PROCESS_EVERY_N_FRAMES == 0:
            face_locations = []
            face_names = []
            names_in_current_frame = set()

            # 1. Detect
            results = detector(rgb_frame, 0)

            for result in results:
                rect = result.rect
                left, top, right, bottom = rect.left(), rect.top(), rect.right(), rect.bottom()
                face_locations.append((left, top, right, bottom))

                # 2. Get Landmarks (Original Frame)
                shape = predictor(rgb_frame, rect)

                # 3. *** NEW STEP: Align Face ***
                # We warp the face so eyes are horizontal and centered
                aligned_face = align_face(rgb_frame, shape)

                # 4. *** NEW STEP: Re-Calculate Landmarks on Aligned Face ***
                # dlib encoder needs a shape that matches the image.
                # Since we changed the image (warped it), we need a new shape.
                h_aligned, w_aligned = aligned_face.shape[:2]
                aligned_rect = dlib.rectangle(0, 0, w_aligned, h_aligned)
                aligned_shape = predictor(aligned_face, aligned_rect)

                # 5. Encode (using the ALIGNED image and ALIGNED shape)
                face_encoding = np.array(facerec.compute_face_descriptor(aligned_face, aligned_shape))

                # 6. Match
                distances = np.linalg.norm(known_encodings - face_encoding, axis=1)
                min_distance_idx = np.argmin(distances)
                min_distance = distances[min_distance_idx]

                if min_distance < RECOGNITION_THRESHOLD:
                    name = known_names[min_distance_idx]
                    names_in_current_frame.add(name)
                else:
                    name = "Unknown"

                face_names.append(name)

            # --- HISTORY LOGIC ---
            recognition_history.append(names_in_current_frame)
            current_verification_stats = {}

            for name in names_in_current_frame:
                if name == "Unknown":
                    continue

                occurrence_count = sum(1 for frame_set in recognition_history if name in frame_set)
                current_verification_stats[name] = occurrence_count

                if occurrence_count >= REQUIRED_HITS:
                    if mark_attendance(name):
                        newly_marked_in_this_frame.append(name)

        frame_count += 1

        # --- FPS Calculation ---
        fps_frame_counter += 1
        if (time.time() - fps_start_time) > 1:
            fps_display = fps_frame_counter / (time.time() - fps_start_time)
            fps_frame_counter = 0
            fps_start_time = time.time()

        # --- DRAWING ---
        for (left, top, right, bottom), name in zip(face_locations, face_names):
            scale = 1 / ratio
            left, top, right, bottom = int(left * scale), int(top * scale), int(right * scale), int(bottom * scale)

            if name == "Unknown":
                color = (0, 0, 255)
                label = name
            elif name in present_students:
                color = (0, 255, 0)
                label = f"{name} (Present)"
            else:
                color = (0, 255, 255)
                hits = current_verification_stats.get(name, 0)
                label = f"{name} {hits}/{REQUIRED_HITS}"

            cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
            cv2.rectangle(frame, (left, bottom - 35), (right, bottom), color, cv2.FILLED)
            cv2.putText(frame, label, (left + 6, bottom - 6), cv2.FONT_HERSHEY_DUPLEX, 0.6, (0, 0, 0), 1)

        if newly_marked_in_this_frame:
            for name in newly_marked_in_this_frame:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                filename = f"{name}_{timestamp}.jpg"
                filepath = os.path.join(SCREENSHOT_DIR, filename)
                cv2.imwrite(filepath, frame)
                print(f"[SNAPSHOT] Saved evidence: {filepath}")

        cv2.putText(frame, f"FPS: {fps_display:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        cv2.imshow('Face Recognition (GPU)', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    video_capture.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    if os.path.exists(ENCODINGS_FILE):
        run_inference()
    else:
        print(f"[ERROR] {ENCODINGS_FILE} not found.")

[INFO] Loading encodings and models...
[INFO] CUDA is active. Using GPU.
[INFO] Starting video stream...
